John S. Anvik, 2025  
University of Lethbridge Student ID: 001224816

### Import Packages

In [1]:
# Import packages
library(tidyverse)

options(repr.matrix.max.cols = Inf) # display all columns in tibbles

Warning message:
"package 'tidyverse' was built under R version 4.4.3"


Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'tibble' was built under R version 4.4.3"
Warning message:
"package 'tidyr' was built under R version 4.4.3"
Warning message:
"package 'readr' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
Warning message:
"package 'dplyr' was built under R version 4.4.3"
Warning message:
"package 'forcats' was built under R version 4.4.3"
Warning message:
"package 'lubridate' was built under R version 4.4.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   4.0.0     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<ht

### Load and Trim Dataset

In [2]:
# minizord.csv already removed non-songs, post-dawn, unmeasureable songs, acoustic outliers
# see "Data Preprocessing/01 preprocessing.ipynb"
data <- read_csv("../data/minizord.csv")

# Only look at North data for now
data <- data %>% filter(neighbourhood == "north") %>% ungroup()

Rows: 11694 Columns: 52
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (4): ID, IST, neighbourhood, Location_Number
dbl  (45): time_rel_sunrise, UST, D_M1_M2, D_M1_M3, D_M1_M4, D_M1_M5, D_M1_M...
lgl   (1): RepeatST
dttm  (1): DateTime
date  (1): date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


### Reformat Data: Row for each possible matching opportunity of focal male

In [3]:
# Just focusing on one focal male for now (central male LbOW)
focal_id <- "LbOW"
focal_match_data <- data

In [4]:
ids <- c("LbOW", "RbDgY", "YKY", "TbTbLg", "DgTbO", "RRO") # Ordered list of IDs correstponding to M1..M6
for (i in 1:6) {
    print(ids[[i]])
    for (d in unique(na.omit(filter(data, ID==ids[[i]])$date))) {
        print(d)
        print(nrow(filter(data, ID == ids[[i]], date == d)))
    }
}

[1] "LbOW"
[1] 19125
[1] 80
[1] 19126
[1] 170
[1] 19128
[1] 5
[1] 19129
[1] 230
[1] 19130
[1] 224
[1] "RbDgY"
[1] 19128
[1] 214
[1] 19129
[1] 224
[1] 19130
[1] 221
[1] "YKY"
[1] 19125
[1] 172
[1] 19126
[1] 191
[1] 19128
[1] 259
[1] 19129
[1] 238
[1] 19130
[1] 212
[1] "TbTbLg"
[1] 19125
[1] 169
[1] 19126
[1] 169
[1] 19128
[1] 164
[1] 19129
[1] 112
[1] 19130
[1] 142
[1] "DgTbO"
[1] 19125
[1] 202
[1] 19126
[1] 233
[1] 19128
[1] 241
[1] 19129
[1] 260
[1] 19130
[1] 233
[1] "RRO"
[1] 19125
[1] 65
[1] 19126
[1] 158
[1] 19128
[1] 136
[1] 19129
[1] 125
[1] 19130
[1] 54


In [5]:
print(unique(na.omit(data$ID)))

[1] "DgTbO"  "LbOW"   "RRO"    "RbDgY"  "TbTbLg" "YKY"   


In [6]:
# Make column for time since Male N's last song
# NA values at top of each day until that male sings
ids <- c("LbOW", "RbDgY", "YKY", "TbTbLg", "DgTbO", "RRO") # Ordered list of IDs correstponding to M1..M6
for (i in 1:6) {
    print(ids[[i]])
    time_last_song <- sprintf("time_M%i_last_song", i)
    time_since_last_song <- sprintf("time_since_M%i_last_song", i)
    print(time_last_song)
    focal_match_data <- focal_match_data %>%
    arrange(DateTime) %>%
    mutate("{time_last_song}" := if_else(ID == ids[[i]], time_rel_sunrise, NA)) %>%
    group_by(date) %>%
    fill({time_last_song}, .direction = "down") %>%
    mutate("{time_since_last_song}" := time_rel_sunrise - !!sym({time_last_song})) %>%
    ungroup()
}

[1] "LbOW"
[1] "time_M1_last_song"
[1] "RbDgY"
[1] "time_M2_last_song"
[1] "YKY"
[1] "time_M3_last_song"
[1] "TbTbLg"
[1] "time_M4_last_song"
[1] "DgTbO"
[1] "time_M5_last_song"
[1] "RRO"
[1] "time_M6_last_song"


In [ ]:
# Male i's nth last song
for (n in 1:2) {
        for (i in 1:6) {
        n_last_song <- sprintf("M%i_%i_last_UST", i, n)
        time_last_song <- sprintf("time_M%i_%i_last_song", i, n)
        time_since_last_song <- sprintf("time_since_M%i_%i_last_song", i, n)
        focal_match_data <- focal_match_data %>%
        # Sort by ID and DateTime
                group_by(date, ID) %>%
                arrange(DateTime) %>%
        # Take the Nth last UST, set it to MX_Nth_last_UST
                mutate("{n_last_song}" := if_else(ID == ids[[i]], lag(UST, n), NA)) %>%
                ungroup() %>% 
        # Sort by DateTime, forward fill
                group_by(date) %>%
                arrange(DateTime) %>%
                fill({n_last_song}, .direction = "down") %>%
        # Calculate time of and since Nth last song
                mutate("{time_last_song}" := if_else(ID == ids[[i]], time_rel_sunrise, NA)) %>%
                group_by(date) %>%
                fill({time_last_song}, .direction = "down") %>%
                 mutate("{time_since_last_song}" := time_rel_sunrise - !!sym({time_last_song})) %>%
                ungroup()
        }
}

In [ ]:
# n_max <- 3  # change as needed

# for (i in seq_along(ids)) {
#   id_name <- ids[i]
  
#   for (n in 1:n_max) {
#     col_name <- sprintf("DLM%i_%i_last_UST", i, n)  # e.g., DLM1_1_last_UST
    
#     focal_match_data <- focal_match_data %>%
#       arrange(DateTime) %>%
#       group_by(ID) %>%
#       # Only apply lag to the specific ID
#       mutate("{col_name}" := if_else(ID == id_name, lag(UST, n), NA_real_)) %>%
#       ungroup() %>%
#       arrange(DateTime) %>%
#       group_by(date) %>%
#       # Fill downward within date, but only for that ID
#       mutate("{col_name}" := if_else(ID == id_name, fill({{col_name}}, .direction = "down"), !!sym(col_name))) %>%
#       ungroup()
#   }
# }

In [ ]:
# # Male i's nth last song
# for (n in 1:2) {
#         for (i in 1:6) {
#                 n_last_song <- sprintf("M%i_%i_last_UST", i, n)
#                 time_last_song <- sprintf("time_M%i_%i_last_song", i, n)
#                 time_since_last_song <- sprintf("time_since_M%i_%i_last_song", i, n)
#                 focal_match_data <- focal_match_data %>%
#                 group_by(ID) %>%
#                 arrange(DateTime) %>%  # important: ensures chronological order within each ID
#                 mutate("{n_last_song}" := lag(UST, n)) %>%
#                 ungroup() %>%
#                 arrange(DateTime) %>%
#                 group_by(date) %>%
#                 fill({n_last_song}, .direction = "down") %>%
#                 ungroup()
#         }
# }
# # View(focal_match_data)

In [ ]:
# focal_match_data <- focal_match_data %>%
#   arrange(DateTime) %>%
#   group_by(date) %>%
  
#   mutate(DLM1_last_UST = if_else(ID == "LbOW", UST, NA))  %>%
#   fill(DLM1_last_UST) %>%

#   mutate(DLM2_last_UST = if_else(ID == "RbDgY", UST, NA))  %>%
#   fill(DLM2_last_UST) %>%

#   mutate(DLM3_last_UST = if_else(ID == "YKY", UST, NA))  %>%
#   fill(DLM3_last_UST) %>%

#   mutate(DLM4_last_UST = if_else(ID == "TbTbLg", UST, NA))  %>%
#   fill(DLM4_last_UST) %>%
  
#   mutate(DLM5_last_UST = if_else(ID == "DgTbO", UST, NA))  %>%
#   fill(DLM5_last_UST) %>%

#   mutate(DLM6_last_UST = if_else(ID == "RRO", UST, NA))  %>%
#   fill(DLM6_last_UST) 

In [10]:
# library(dplyr)
# library(tidyr)
# library(purrr)

# ids <- c("LbOW", "RbDgY", "YKY", "TbTbLg", "DgTbO", "RRO")

# focal_match_data <- focal_match_data %>%
#   arrange(date, DateTime) %>%
#   group_by(date) %>%
#   {
#     reduce(ids, function(df, id) {
#       col_name <- paste0("DLM", which(ids == id), "_second_last_UST")
      
#       df %>%
#         # For each date, track the rolling last and second-last USTs for this ID
#         mutate(
#           temp_match = if_else(ID == id, UST, NA_real_),
#           # cumulative lists of non-NA values for this ID
#           temp_list = accumulate(temp_match, function(acc, x) {
#             if (!is.na(x)) c(acc, x) else acc
#           }, .init = numeric())[ -1 ],
#           temp_second_last = map_dbl(temp_list, ~ if (length(.x) >= 2) .x[length(.x) - 1] else NA_real_)
#         ) %>%
#         select(-temp_match, -temp_list) %>%
#         rename(!!col_name := temp_second_last)
#     }, .init = .)
#   }

In [11]:
tail(focal_match_data)

ID,DateTime,date,time_rel_sunrise,UST,IST,D_M1_M2,D_M1_M3,D_M1_M4,D_M1_M5,D_M1_M6,D_M2_M3,D_M2_M4,D_M2_M5,D_M2_M6,D_M3_M4,D_M3_M5,D_M3_M6,D_M4_M5,D_M4_M6,D_M5_M6,M1_Lat,M1_Lon,M2_Lat,M2_Lon,M3_Lat,M3_Lon,M4_Lat,M4_Lon,M5_Lat,M5_Lon,M6_Lat,M6_Lon,M1_last_UST,M2_last_UST,M3_last_UST,M4_last_UST,M5_last_UST,M6_last_UST,M1_match,M2_match,M3_match,M4_match,M5_match,M6_match,number_of_matches,does_it_match,neighbourhood,Location_Number,Time_Diff,RepeatST,Repeat_ST_Run,time_M1_last_song,time_since_M1_last_song,time_M2_last_song,time_since_M2_last_song,time_M3_last_song,time_since_M3_last_song,time_M4_last_song,time_since_M4_last_song,time_M5_last_song,time_since_M5_last_song,time_M6_last_song,time_since_M6_last_song,M1_1_last_UST,M2_1_last_UST,M3_1_last_UST,M4_1_last_UST,M5_1_last_UST,M6_1_last_UST,M1_2_last_UST,M2_2_last_UST,M3_2_last_UST,M4_2_last_UST,M5_2_last_UST,M6_2_last_UST,DLM1_last_UST,DLM2_last_UST,DLM3_last_UST,DLM4_last_UST,DLM5_last_UST,DLM6_last_UST
<chr>,<dttm>,<date>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
DgTbO,2022-05-18 05:49:27,2022-05-18,-332.39960,10,DgTbO_3,97.32587,39.73681,67.67739,60.4591,53.6376,60.45412,42.09414,151.8528,127.82773,43.97431,99.32688,69.72815,114.3889,111.6779,87.49532,17.98071,-67.1644,17.98140,-67.16383,17.98092,-67.16409,17.9813,-67.16421,17.98053,-67.16493,17.98029,-67.16415,12,19,166,62,10,10,0,0,0,0,0,1,1,1,north,Y.001,15,FALSE,1,-416.9097,84.51011,-350.85736,18.457761,-505.4082,173.0086,-487.5595,155.1599,-332.3996,0.00000,-1166.749,834.3490,8,8,8,8,8,8,8,8,8,8,8,8,12,19,166,62,10,10
RbDgY,2022-05-18 05:49:38,2022-05-18,-321.84534,8,RbDgY_2,94.03680,39.73681,67.67739,60.4591,53.6376,54.40540,62.24870,153.6539,110.10557,43.97431,99.32688,69.72815,114.3889,111.6779,87.49532,17.98071,-67.1644,17.98116,-67.16364,17.98092,-67.16409,17.9813,-67.16421,17.98053,-67.16493,17.98029,-67.16415,12,8,166,62,10,10,0,0,0,0,0,0,0,0,north,28,29,FALSE,1,-416.9097,95.06437,-321.84534,0.000000,-505.4082,183.5628,-487.5595,165.7142,-332.3996,10.55426,-1166.749,844.9033,19,19,19,19,19,19,3,3,3,3,3,3,12,8,166,62,10,10
DgTbO,2022-05-18 05:49:39,2022-05-18,-320.61551,10,DgTbO_3,94.03680,39.73681,67.67739,60.4591,53.6376,54.40540,62.24870,153.6539,110.10557,43.97431,99.32688,69.72815,114.3889,111.6779,87.49532,17.98071,-67.1644,17.98116,-67.16364,17.98092,-67.16409,17.9813,-67.16421,17.98053,-67.16493,17.98029,-67.16415,12,8,166,62,10,10,0,0,0,0,0,1,1,1,north,Y.001,12,TRUE,2,-416.9097,96.29419,-321.84534,1.229823,-505.4082,184.7927,-487.5595,166.9440,-320.6155,0.00000,-1166.749,846.1331,10,10,10,10,10,10,8,8,8,8,8,8,12,8,166,62,10,10
RbDgY,2022-05-18 05:49:51,2022-05-18,-308.70611,3,RbDgY_1,94.03680,39.73681,67.67739,60.4591,53.6376,54.40540,62.24870,153.6539,110.10557,43.97431,99.32688,69.72815,114.3889,111.6779,87.49532,17.98071,-67.1644,17.98116,-67.16364,17.98092,-67.16409,17.9813,-67.16421,17.98053,-67.16493,17.98029,-67.16415,12,3,166,62,10,10,0,0,0,0,0,0,0,0,north,28,13,FALSE,1,-416.9097,108.20359,-308.70611,0.000000,-505.4082,196.7021,-487.5595,178.8534,-320.6155,11.90940,-1166.749,858.0425,8,8,8,8,8,8,19,19,19,19,19,19,12,3,166,62,10,10
DgTbO,2022-05-18 05:51:27,2022-05-18,-212.08774,25,DgTbO_13,78.37111,39.73681,67.67739,60.4591,53.6376,41.23076,65.91978,138.8283,87.47313,43.97431,99.32688,69.72815,114.3889,111.6779,87.49532,17.98071,-67.1644,17.98096,-67.16370,17.98092,-67.16409,17.9813,-67.16421,17.98053,-67.16493,17.98029,-67.16415,12,3,166,62,25,10,0,0,0,0,0,0,0,0,north,Y.001,108,FALSE,1,-416.9097,204.82197,-308.70611,96.618378,-505.4082,293.3204,-487.5595,275.4718,-212.0

In [12]:
# Focal Male's song-type repertoire
focal_repertoire <- unique(filter(data, ID == focal_id)$UST)
print("Focal Male's Song Type Repertoire:")
focal_repertoire

# Select only rows with focal male's songs
focal_match_data <- focal_match_data %>%
    filter(ID == focal_id) %>%
# Create column for M1's last UST
    # mutate(M1_last_UST = lag(UST)) %>%
# Create column for time of and time since M1's last UST    
    # mutate(time_M1_last_song = lag(time_rel_sunrise)) %>%
    # mutate(time_since_M1_last_song = time_rel_sunrise - time_M1_last_song) %>%
# Input variables: M1-M6's last UST, time since M1-M6 UST, # M2-M6's distance
    select(
        DateTime, time_rel_sunrise,
        "D_M1_M2", "D_M1_M3", "D_M1_M4", "D_M1_M5", "D_M1_M6",
        UST,
        "M1_last_UST", "M2_last_UST", "M3_last_UST", "M4_last_UST", "M5_last_UST", "M6_last_UST",
        
        contains("last_UST"), # Nth last song
        contains("last_song"), # Time since last song
        # time_M1_last_song, time_M2_last_song, time_M3_last_song, time_M4_last_song, time_M5_last_song, time_M6_last_song,
    	time_since_M1_last_song, time_since_M2_last_song, time_since_M3_last_song, time_since_M4_last_song, time_since_M5_last_song, time_since_M6_last_song, # did not include time since M1 last song because it is always 0
        number_of_matches, does_it_match
        # Also consider adding "streak"
    ) %>% 
    mutate(across(contains("UST"), ~ as.factor(.x)), 
        does_it_match = as.factor(does_it_match),
        number_of_matches = as.integer(number_of_matches)) #%>%
# Set all neighbours' songs that aren't in M1's repertoire to 0
    # mutate(across(contains("UST"), ~ if_else (.x %in% focal_repertoire, .x, as.character(0))))

[1] "Focal Male's Song Type Repertoire:"


[1]  10   3   8   9  13  29   5  16  21  NA  20 120  17 117 115 116 118 119  12

Adding missing grouping variables: `date`


In [13]:
# Compared to M1's repertoire, which songs are in the other males' repertoires?
print("Male 1's Repertoire:")
print(focal_repertoire)
for (i in 2:6) {
    print(sprintf("Male %i's Repertoire:", i))
    print(pull(unique(focal_match_data %>% select({sprintf("M%i_last_UST", i)}))))
    # print("\n")
}


[1] "Male 1's Repertoire:"
 [1]  10   3   8   9  13  29   5  16  21  NA  20 120  17 117 115 116 118 119  12
[1] "Male 2's Repertoire:"


Adding missing grouping variables: `date`


 [1] <NA> <NA> 19   8    9    18   3    8    9    21   13   22   5    16   20  
[16] 17   90   27   18   92   3    17   21   13   19   8    5    90   20   18  
[31] 16   27   91   1    22   9    92  
Levels: 8 9 18 19 3 5 13 16 17 20 21 22 27 90 92 1 91
[1] "Male 3's Repertoire:"


Adding missing grouping variables: `date`


 [1] 165  10   3    8    166  35   13   16   167  30   22   <NA> 1    3    8   
[16] 17   166  10   19   18   169  27   20   9    5    21   16   165  13   167 
[31] 170  30   27   20   3    166  18   <NA> 3    166  8    9    10   13   21  
[46] 16   1    5    165  17   169  27   19   18   172  <NA> 10   19   1    8   
[61] 21   165  13   166  169  27   18   16   5    9    20   3    17  
23 Levels: 3 8 10 13 16 22 30 35 165 166 167 1 5 9 17 18 19 20 21 27 ... 172
[1] "Male 4's Repertoire:"


Adding missing grouping variables: `date`


 [1] 13   61   10   9    60   8    3    16   22   21   5    20   18   1    <NA>
[16] 8    18   9    60   19   62   10   3    16   22   17   1    13   20   5   
[31] 21   62   19   9    <NA> 9    3    8    10   22   16   1    60   5    21  
[46] 13   20   17   64   18   19   62   61   <NA> 62   10   13   19   16   1   
[61] 5    3    18   64   17   22   21   9    61   20   8    60  
Levels: 1 3 5 8 9 10 13 16 18 20 21 22 60 61 17 19 62 64
[1] "Male 5's Repertoire:"


Adding missing grouping variables: `date`


 [1] 20   33   1    7    153  10   8    15   25   32   33   1    153  7    20  
[16] 12   8    10   11   154  4    33   11   8    153  <NA> 33   32   7    4   
[31] 6    11   10   20   12   8    1    153  15   153  20   7    6    33   32  
[46] 11   8    1    155  10   15   12   4   
Levels: 1 7 8 10 15 20 25 33 153 4 11 12 32 154 6 155
[1] "Male 6's Repertoire:"


Adding missing grouping variables: `date`


 [1] 98   12   8    3    9    97   13   10   <NA> 98   10   8    13   9    17  
[16] 12   30   97   3    5    13   <NA> 5    97   8    3    30   9    28   12  
[31] 10   98   13   <NA> 10   12   98   97   13   19   30   5    3    9    8   
Levels: 3 8 9 10 12 13 97 98 5 17 30 28 19


In [14]:
head(focal_match_data)

date,DateTime,time_rel_sunrise,D_M1_M2,D_M1_M3,D_M1_M4,D_M1_M5,D_M1_M6,UST,M1_last_UST,M2_last_UST,M3_last_UST,M4_last_UST,M5_last_UST,M6_last_UST,M1_1_last_UST,M2_1_last_UST,M3_1_last_UST,M4_1_last_UST,M5_1_last_UST,M6_1_last_UST,M1_2_last_UST,M2_2_last_UST,M3_2_last_UST,M4_2_last_UST,M5_2_last_UST,M6_2_last_UST,DLM1_last_UST,DLM2_last_UST,DLM3_last_UST,DLM4_last_UST,DLM5_last_UST,DLM6_last_UST,time_M1_last_song,time_since_M1_last_song,time_M2_last_song,time_since_M2_last_song,time_M3_last_song,time_since_M3_last_song,time_M4_last_song,time_since_M4_last_song,time_M5_last_song,time_since_M5_last_song,time_M6_last_song,time_since_M6_last_song,number_of_matches,does_it_match
<date>,<dttm>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<fct>
2022-05-13,2022-05-13 05:37:08,-1161.593,NA,52.93973,67.67739,62.95599,53.6376,10,10,NA,165,13,20,98,7,7,7,7,7,7,10,10,10,10,10,10,10,NA,165,13,20,98,-1161.593,0,NA,NA,-1167.065,5.472131,-1167.219,5.626019,-1166.300,4.7073998,-1341.908,180.3157,0,0
2022-05-13,2022-05-13 05:37:21,-1148.784,NA,46.47336,67.67739,62.95599,53.6376,10,10,NA,10,61,33,98,10,10,10,10,10,10,153,153,153,153,153,153,10,NA,10,61,153,98,-1148.784,0,NA,NA,-1149.934,1.149475,-1153.090,4.305930,-1148.543,-0.2417286,-1341.908,193.1240,1,1
2022-05-13,2022-05-13 05:37:29,-1140.450,NA,46.47336,67.67739,62.95599,53.6376,3,3,NA,10,10,33,98,10,10,10,10,10,10,10,10,10,10,10,10,3,NA,10,10,33,98,-1140.450,0,NA,NA,-1149.934,9.483614,-1148.608,8.158114,-1141.642,1.1915795,-1341.908,201.4581,0,0
2022-05-13,2022-05-13 05:37:39,-1130.582,NA,46.47336,67.67739,62.95599,53.6376,8,8,NA,3,9,1,98,3,3,3,3,3,3,10,10,10,10,10,10,8,NA,3,9,1,98,-1130.582,0,NA,NA,-1132.836,2.253478,-1131.712,1.129353,-1136.170,5.5875481,-1341.908,211.3260,0,0
2022-05-13,2022-05-13 05:37:47,-1122.382,NA,46.47336,67.67739,62.95599,53.6376,9,9,NA,8,60,7,98,8,8,8,8,8,8,3,3,3,3,3,3,9,NA,8,60,7,98,-1122.382,0,NA,NA,-1127.425,5.042585,-1125.170,2.788003,-1123.632,1.2496293,-1341.908,219.5263,0,0
2022-05-13,2022-05-13 05:38:01,-1108.298,NA,46.47336,67.67739,62.95599,53.6376,13,13,NA,166,8,153,98,9,9,9,9,9,9,8,8,8,8,8,8,13,NA,166,8,153,98,-1108.298,0,NA,NA,-1111.024,2.726158,-1116.499,8.201041,-1112.026,3.7281486,-1341.908,233.6108,0,0


### Export Data

In [15]:
write_csv(focal_match_data, "../data/matching_opportunities.csv")

ERROR: Error: Cannot open file for writing:
* 'C:/Users/johna/OneDrive/Documents/Birdsong/data/matching_opportunities.csv'
